# Bài tập thực hành Optimization: CNN trên CIFAR-10



## 0. Cài đặt thư viện xuất PDF


In [ ]:
!pip -q install reportlab

## 1. Import và cấu hình thí nghiệm

In [ ]:
import os
import random
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from IPython.display import display
from torch.utils.data import DataLoader, Subset, random_split


GITHUB_LINK = "https://github.com/huybitvvt/Ex_HocMay_NguyenDoanHuy.git"

FAST_DEV_RUN = False

EPOCHS = 4 if FAST_DEV_RUN else 20
BATCH_SIZE = 128
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Epochs:", EPOCHS, "| Fast dev run:", FAST_DEV_RUN)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


set_seed(SEED)
os.makedirs("outputs", exist_ok=True)

## 2. Chuẩn bị dataset CIFAR-10

Training set có augmentation nhẹ (`RandomCrop`, `RandomHorizontalFlip`). Validation set chỉ normalize để đánh giá ổn định.

In [ ]:
def build_loaders(batch_size=128, fast_dev_run=True):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])
    transform_val = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])

    train_full = torchvision.datasets.CIFAR10(root="data", train=True, download=True, transform=transform_train)
    val_source = torchvision.datasets.CIFAR10(root="data", train=True, download=True, transform=transform_val)

    indices = list(range(len(train_full)))
    train_idx, val_idx = random_split(indices, [45000, 5000], generator=torch.Generator().manual_seed(SEED))
    train_set = Subset(train_full, train_idx.indices)
    val_set = Subset(val_source, val_idx.indices)

    if fast_dev_run:
        train_set = Subset(train_set, range(10000))
        val_set = Subset(val_set, range(2000))

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader


train_loader, val_loader = build_loaders(BATCH_SIZE, FAST_DEV_RUN)
print("Train batches:", len(train_loader), "| Validation batches:", len(val_loader))

## 3. Xây dựng mô hình CNN

Cùng một kiến trúc CNN được dùng cho các thí nghiệm. Ta bật/tắt Batch Normalization và thay đổi Dropout bằng tham số để so sánh công bằng hơn.

In [ ]:
class CIFAR10CNN(nn.Module):
    def __init__(self, use_batch_norm=False, dropout=0.0):
        super().__init__()

        def conv_block(in_channels, out_channels):
            layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)]
            if use_batch_norm:
                layers.append(nn.BatchNorm2d(out_channels))
            layers += [nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)]
            if use_batch_norm:
                layers.append(nn.BatchNorm2d(out_channels))
            layers += [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

## 4. Hàm train, evaluate và cấu hình optimizer

In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    use_batch_norm: bool
    dropout: float
    optimizer_name: str
    lr: float
    momentum: float = 0.0


def make_optimizer(model, cfg):
    if cfg.optimizer_name == "SGD":
        return optim.SGD(model.parameters(), lr=cfg.lr)
    if cfg.optimizer_name == "SGD + Momentum":
        return optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum)
    if cfg.optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=cfg.lr)
    raise ValueError(f"Unknown optimizer: {cfg.optimizer_name}")


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def train_one_experiment(cfg, train_loader, val_loader, epochs):
    set_seed(SEED)
    model = CIFAR10CNN(cfg.use_batch_norm, cfg.dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, cfg)
    history = []
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total = 0.0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)

        train_loss = total_loss / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append({
            "experiment": cfg.name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })
        print(f"{cfg.name:18s} | epoch {epoch:02d}/{epochs} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    elapsed = time.time() - start
    hist_df = pd.DataFrame(history)
    best_val_acc = hist_df["val_accuracy"].max()

    # Epoch convergence: epoch đầu tiên đạt >= 99% accuracy tốt nhất của chính cấu hình đó.
    threshold = 0.99 * best_val_acc
    convergence_epoch = int(hist_df.loc[hist_df["val_accuracy"] >= threshold, "epoch"].iloc[0])

    summary = {
        "experiment": cfg.name,
        "optimizer": cfg.optimizer_name,
        "batch_norm": "Có" if cfg.use_batch_norm else "Không",
        "dropout": cfg.dropout,
        "final_train_loss": hist_df["train_loss"].iloc[-1],
        "best_val_accuracy": best_val_acc,
        "final_val_accuracy": hist_df["val_accuracy"].iloc[-1],
        "train_time_sec": elapsed,
        "convergence_epoch": convergence_epoch,
    }
    return summary, hist_df

## 5. B1 và B2: Chạy các thí nghiệm

Các cấu hình gồm:

1. CNN cơ bản + SGD.
2. CNN + BatchNorm + Dropout 0.2 + SGD Momentum.
3. CNN + BatchNorm + Dropout 0.5 + SGD Momentum.
4. CNN + BatchNorm + Dropout 0.2 + Adam.
5. CNN + BatchNorm + Dropout 0.5 + Adam.

In [ ]:
configs = [
    ExperimentConfig("B1_Basic_SGD", False, 0.0, "SGD", 0.01),
    ExperimentConfig("B2_BN_DO02_SGDM", True, 0.2, "SGD + Momentum", 0.01, 0.9),
    ExperimentConfig("B2_BN_DO05_SGDM", True, 0.5, "SGD + Momentum", 0.01, 0.9),
    ExperimentConfig("B2_BN_DO02_Adam", True, 0.2, "Adam", 0.001),
    ExperimentConfig("B2_BN_DO05_Adam", True, 0.5, "Adam", 0.001),
]

summaries = []
histories = []

for cfg in configs:
    print("\n===", cfg.name, "===")
    summary, history = train_one_experiment(cfg, train_loader, val_loader, EPOCHS)
    summaries.append(summary)
    histories.append(history)

results_df = pd.DataFrame(summaries).sort_values("best_val_accuracy", ascending=False).reset_index(drop=True)
history_df = pd.concat(histories, ignore_index=True)

results_df.to_csv("optimization_results.csv", index=False)
history_df.to_csv("training_history.csv", index=False)

display(results_df)

## 6. B3: Bảng so sánh và biểu đồ

In [ ]:
display_cols = [
    "experiment", "optimizer", "batch_norm", "dropout",
    "final_train_loss", "best_val_accuracy", "final_val_accuracy",
    "train_time_sec", "convergence_epoch"
]

comparison_table = results_df[display_cols].copy()
comparison_table["final_train_loss"] = comparison_table["final_train_loss"].round(4)
comparison_table["best_val_accuracy"] = (comparison_table["best_val_accuracy"] * 100).round(2)
comparison_table["final_val_accuracy"] = (comparison_table["final_val_accuracy"] * 100).round(2)
comparison_table["train_time_sec"] = comparison_table["train_time_sec"].round(1)

display(comparison_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, group in history_df.groupby("experiment"):
    axes[0].plot(group["epoch"], group["train_loss"], marker="o", label=name)
    axes[1].plot(group["epoch"], group["val_accuracy"] * 100, marker="o", label=name)

axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. B4: Kết luận tự động từ kết quả

In [ ]:
best = results_df.iloc[0]

conclusion = f"""
Sau khi chạy các thí nghiệm, em thấy cấu hình cho kết quả tốt nhất là {best['experiment']},
sử dụng optimizer {best['optimizer']}, Batch Normalization: {best['batch_norm']},
và Dropout = {best['dropout']}.

Em chọn cấu hình này vì nó đạt validation accuracy cao nhất ({best['best_val_accuracy'] * 100:.2f}%).
Training loss cuối là {best['final_train_loss']:.4f}, mô hình gần đạt kết quả tốt nhất từ epoch
{int(best['convergence_epoch'])}, và tổng thời gian train là {best['train_time_sec']:.1f} giây.

So với mô hình CNN cơ bản dùng SGD, các cấu hình có Batch Normalization học ổn định hơn.
Dropout giúp hạn chế overfitting, nhưng nếu dropout quá cao thì mô hình có thể học chậm hơn.
Adam thường cải thiện nhanh trong những epoch đầu, còn SGD + Momentum vẫn là một lựa chọn tốt
nếu learning rate được chọn phù hợp.

Vì vậy, mô hình tốt nhất không chỉ là mô hình có accuracy cao nhất, mà còn cần có loss giảm ổn định,
khả năng tổng quát hóa tốt trên tập validation và thời gian train hợp lý.
"""

print(conclusion)

## 8. Tải file kết quả từ Colab

In [ ]:
from google.colab import files

for file_name in ["optimization_results.csv", "training_history.csv", "training_curves.png", "optimization_cifar10_report.pdf"]:
    if os.path.exists(file_name):
        files.download(file_name)